In [0]:
from pyspark.sql import SparkSession
import os

def generate_mock_data():
    print(" Generating Mock Data for Medallion Pipeline...")
    
    # In Databricks, 'spark' is global. If running locally, this creates a session.
    try:
        active_spark = spark
    except NameError:
        active_spark = SparkSession.builder.appName("DataBricks").getOrCreate()

    # Use Unity Catalog volume instead of /tmp (DBFS root is disabled)
    base_path = "/Volumes/workspace/default/medallion_data"

    # ==========================================
    # 📦 DAY 1: THE INITIAL LOAD
    # ==========================================
    print(f"Creating Day 1 Data at {base_path}/raw_day1/...")

    # 1. Customers (Day 1) - JSON
    customers_day1 = [
        ("C-100", "alice smith", "New York", "Basic", "2024-01-01T08:00:00Z"),
        ("C-101", "BOB JONES", "London", "Premium", "2024-01-01T09:30:00Z"),
        ("C-102", "charlie brown", "Chicago", "Basic", "2024-01-02T11:15:00Z")
    ]
    df_cust_d1 = active_spark.createDataFrame(customers_day1, ["customer_id", "name", "city", "tier", "event_ts"])
    df_cust_d1.write.mode("overwrite").json(f"{base_path}/raw_day1/customers")

    # 2. Products (Day 1) - JSON
    products_day1 = [
        ("P-01", "Wireless Mouse", "Electronics", "25.00"),
        ("P-02", "Mechanical Keybord", "Elect", "150.00"), # Typo in category
        ("P-03", "Desk Mat", "Office", "null")           # Missing price
    ]
    df_prod_d1 = active_spark.createDataFrame(products_day1, ["product_id", "product_name", "category", "price"])
    df_prod_d1.write.mode("overwrite").json(f"{base_path}/raw_day1/products")

    # 3. Orders (Day 1) - CSV (Messy dates and duplicates)
    orders_day1 = [
        ("ORD-9001", "C-100", "P-01", "2024-01-05", "25.00", "Shipped"),
        ("ORD-9002", "c-101", "P-02", "01/06/2024", "150.00", "pending"),
        ("ORD-9003", "C-102", "P-03", "2024-01-06T14:30:00", "abc", "Cancelled"),
        ("ORD-9001", "C-100", "P-01", "2024-01-05", "25.00", "Shipped") # Duplicate
    ]
    df_ord_d1 = active_spark.createDataFrame(orders_day1, ["order_id", "customer_id", "product_id", "order_date", "amount", "status"])
    df_ord_d1.write.mode("overwrite").option("header", "true").csv(f"{base_path}/raw_day1/orders")


    # ==========================================
    #  DAY 2: THE INCREMENTAL LOAD (CDC)
    # ==========================================
    print(f"Creating Day 2 Data at {base_path}/raw_day2/...")

    # 1. Customers (Day 2) - Triggers SCD Type 2
    customers_day2 = [
        ("C-100", "Alice Smith", "Los Angeles", "Premium", "2024-02-01T10:00:00Z"), # Moved & Upgraded!
        ("C-103", "Diana Prince", "Paris", "Premium", "2024-02-01T12:00:00Z")       # Brand new customer
    ]
    df_cust_d2 = active_spark.createDataFrame(customers_day2, ["customer_id", "name", "city", "tier", "event_ts"])
    df_cust_d2.write.mode("overwrite").json(f"{base_path}/raw_day2/customers")

    # 2. Products (Day 2) - Triggers SCD Type 1
    products_day2 = [
        ("P-02", "Mechanical Keyboard", "Electronics", "145.00"), # Fixed typo, updated price
        ("P-04", "Monitor", "Electronics", "300.00")              # New product
    ]
    df_prod_d2 = active_spark.createDataFrame(products_day2, ["product_id", "product_name", "category", "price"])
    df_prod_d2.write.mode("overwrite").json(f"{base_path}/raw_day2/products")

    # 3. Orders (Day 2)
    orders_day2 = [
        ("ORD-9004", "C-100", "P-04", "2024-02-05", "300.00", "Delivered"),
        ("ORD-9005", "C-103", "P-01", "02/05/2024", "25.00", "Shipped")
    ]
    df_ord_d2 = active_spark.createDataFrame(orders_day2, ["order_id", "customer_id", "product_id", "order_date", "amount", "status"])
    df_ord_d2.write.mode("overwrite").option("header", "true").csv(f"{base_path}/raw_day2/orders")

    print("✅ All Mock Data Generated Successfully!")

if __name__ == "__main__":
    generate_mock_data()